# Modélisation du second tour

L'objectif est de partir des résultats du premier tour pour produire une **distribution de sièges** au second tour. Le modèle ne cherche pas à estimer des taux de report « vrais » : il tire des matrices plausibles à partir d'hypothèses simples, puis mesure les projections qui en découlent.

Nous suivrons d'abord une seule circonscription, la `0101`, avant de répéter exactement le même mécanisme à l'échelle nationale.

In [1]:
import polars as pl

from analyse_legislatives.data import load_full_results
from analyse_legislatives.models import build
from analyse_legislatives.parties import PoliticalFamily
from analyse_legislatives.projections import (
    median_scenario_seats,
    seats_by_simulation,
)
from analyse_legislatives.simulation import run, to_long_frame
from analyse_legislatives.transfers import normalize_for_district
from analyse_legislatives.viz.charts import (
    render_dominant_party_chart,
    render_ternary_chart,
)

SEED = 42
N_SIMUS = 200  # Démonstration rapide, pas estimation précise des quantiles.

## 1. Les informations disponibles après le premier tour

Le chargement construit directement les circonscriptions consommées par le modèle. Il distingue les voix des candidats qualifiés, les voix des candidats éliminés et les suffrages non exprimés. Ces derniers regroupent l'abstention, les bulletins blancs et les bulletins nuls.

In [2]:
first_round = load_full_results()
districts = first_round.districts

district_index = next(
    i for i, district in enumerate(districts)
    if district.circonscription.id == "0101"
)
district = districts[district_index]

In [3]:
print(district)
print(f"Suffrages non exprimés : {district.non_expressed:,}".replace(",", " "))

Résultats 1er tour
=== 1ère circonscription (0101) ===
Partis en lice
LR : 14495
RN+ : 23819
Partis éliminés
NFP+ : 14607
RN+ : 511
ENS+ : 7063
Suffrages non exprimés : 26 348


Dans cette circonscription, le second tour oppose `LR` à `RN+`. Les autres voix forment des réservoirs susceptibles de se reporter vers l'un des deux candidats ou vers les suffrages non exprimés.

## 2. Tirer une matrice de report plausible

Commençons par le modèle `national`. Pour une simulation donnée, il utilise la même matrice de report dans toutes les circonscriptions. Il constitue ainsi le point de départ le plus simple.

In [4]:
model = build("national", seed=SEED)

Les hypothèses de report sont des **ordres partiels** de préférence, stockés dans `config/model.yaml`. Par exemple, pour les électeurs d'`ENS+`, le premier palier est préféré au second, mais le modèle n'impose aucun ordre entre les partis d'un même palier.

In [5]:
model.transfer_orderings[PoliticalFamily.ENSx]

[[<PoliticalFamily.DVG: 'DVG'>,
  <PoliticalFamily.DVD: 'DVD'>,
  <PoliticalFamily.LR: 'LR'>,
  <PoliticalFamily.NFPx: 'NFP+'>],
 [<PoliticalFamily.RNx: 'RN+'>, 'NON_EXPRIMES']]

Une simulation commence par tirer des paramètres nationaux. Parmi eux, `alpha` contrôle la concentration de la Dirichlet symétrique : plus il est faible, plus une ligne peut être déséquilibrée. Le modèle tire aussi un ordre total compatible avec chaque ordre partiel.

In [6]:
draw = model.draw_simulation()
{
    "alpha": round(draw.alpha, 3),
    "rétention des non-exprimés": round(draw.non_expressed_retention, 3),
    "démobilisation des qualifiés": round(draw.qualified_demobilisation, 3),
    "tilt": round(draw.tilt, 3),
}

{'alpha': 0.855,
 'rétention des non-exprimés': 0.739,
 'démobilisation des qualifiés': 0.031,
 'tilt': -0.691}

In [7]:
draw.extensions[PoliticalFamily.ENSx]

[<PoliticalFamily.LR: 'LR'>,
 <PoliticalFamily.DVG: 'DVG'>,
 <PoliticalFamily.NFPx: 'NFP+'>,
 <PoliticalFamily.DVD: 'DVD'>,
 'NON_EXPRIMES',
 <PoliticalFamily.RNx: 'RN+'>]

La Dirichlet produit ensuite une ligne de proportions, puis ses composantes sont classées selon l'ordre total tiré. La matrice suivante couvre encore toutes les destinations possibles à l'échelle nationale.

In [8]:
matrix = model.sample_transfer_matrices([district], draw)[0]
print(matrix)

TransferMatrix
source          NFP+      LR     RN+    ENS+     DIV     DVG     DVD  NON_EXPRIMES
──────────────────────────────────────────────────────────────────────────────────
NFP+               —   10.8%    1.9%   19.6%   21.5%   32.9%    9.5%          3.8%
LR              5.1%       —    5.6%   28.6%    2.0%   19.5%   31.5%          7.6%
RN+            17.5%   25.5%       —    3.0%    9.3%   19.7%   25.0%          0.1%
ENS+           14.7%   47.7%    4.5%       —    2.2%   17.8%    7.7%          5.4%
DIV             6.9%    5.1%    6.5%   24.4%       —   10.9%   45.3%          0.9%
DVG            21.9%   16.1%    1.8%   16.3%   29.5%       —   12.0%          2.4%
DVD             0.5%   21.8%   21.4%   27.3%    5.5%   14.7%       —          8.8%
NON_EXPRIMES       ·       ·       ·       ·       ·       ·       ·         73.9%

own retention: NFP+=96.9%, LR=96.9%, RN+=96.9%, ENS+=96.9%, DIV=96.9%, DVG=96.9%, DVD=96.9%


Chaque ligne de cette matrice somme déjà à 100 %. Il faut cependant l'adapter à la circonscription : les reports vers les partis absents du second tour sont impossibles. Leurs colonnes sont retirées, les probabilités restantes sont renormalisées et les voix des candidats qualifiés peuvent soit rester chez eux, soit se démobiliser.

In [9]:
normalized_matrix = normalize_for_district(
    matrix, district, draw.tilt
)
print(normalized_matrix)

TransferMatrix
source          NFP+      LR     RN+    ENS+     DIV     DVG     DVD  NON_EXPRIMES
──────────────────────────────────────────────────────────────────────────────────
NFP+               /   65.2%   11.5%       /       /       /       /         23.2%
LR                 /   96.9%    0.0%       /       /       /       /          3.1%
RN+                /    0.0%   96.9%       /       /       /       /          3.1%
ENS+               /   82.8%    7.8%       /       /       /       /          9.4%
DIV                /   40.7%   51.9%       /       /       /       /          7.4%
DVG                /   79.3%    9.1%       /       /       /       /         11.6%
DVD                /   41.9%   41.2%       /       /       /       /         16.9%
NON_EXPRIMES       /   15.2%   10.8%       /       /       /       /         73.9%


## 3. Passer des taux de report aux voix

Pour chaque réservoir du premier tour, les voix sont réparties par un tirage multinomial selon la ligne correspondante. Cette dernière étape représente l'aléa individuel autour des taux de report.

In [10]:
prediction = model.predict_circonscription(
    district, matrix, draw.tilt
)
print(prediction)

Résultats pour : 1ère circonscription
Vainqueur : LR
NFP+: 0
LR: 33497
RN+: 28704
ENS+: 0
DIV: 0
DVG: 0
DVD: 0
NON_EXPRIMES: 24642



Ce résultat n'est qu'un scénario possible. Une projection probabiliste consiste à recommencer toute la procédure : nouveaux paramètres nationaux, nouvelle matrice et nouveaux tirages de voix.

## 4. Répéter la simulation à l'échelle nationale

`run` applique ce mécanisme à toutes les circonscriptions restant à pourvoir. Les sièges déjà gagnés au premier tour sont ensuite ajoutés au vainqueur simulé de chaque circonscription.

In [11]:
results = run(model, districts, n_simus=N_SIMUS)
seats = seats_by_simulation(results, first_round.first_round_seats)

In [12]:
median_scenario_seats(seats)

{'DIV': 11,
 'NFP+': 206,
 'DVG': 11,
 'ENS+': 161,
 'LR': 35,
 'DVD': 24,
 'RN+': 129}

In [13]:
render_dominant_party_chart(seats)

alt.LayerChart(...)

Ce graphique décrit la **distribution prédictive a priori** du modèle. Il ne s'agit pas d'une distribution postérieure : aucun résultat du second tour n'a été utilisé pour apprendre les taux de report.

## 5. Revenir à la circonscription 0101

Le diagramme ternaire montre conjointement les voix de `LR`, de `RN+` et les suffrages non exprimés. Chaque point correspond à une simulation nationale différente.

In [14]:
district_frame = (
    to_long_frame(results, districts)
    .filter(pl.col("id_circo") == "0101")
)

render_ternary_chart(
    district_frame,
    str(PoliticalFamily.LR),
    str(PoliticalFamily.RNx),
    "LR / RN+ / Non exprimés — circonscription 0101",
)

alt.LayerChart(...)

Le diamant est le **barycentre** des simulations. Le nuage est plus informatif qu'une moyenne seule : il montre quelles combinaisons de voix et de participation restent compatibles avec les hypothèses du modèle.

## 6. Pour aller plus loin : introduire des variations locales

Le modèle `national` impose les mêmes taux de report partout. Deux variantes relâchent progressivement cette simplification :

- `national_anchored` conserve une matrice nationale, mais ancre les suffrages exprimés sur le premier tour ;
- `kernel_anchored` ajoute des variations locales corrélées selon la similarité des résultats du premier tour.

Comparer ces deux variantes permet d'isoler l'effet de l'hétérogénéité locale : elles partagent le même mécanisme d'ancrage de la participation.

In [15]:
def seat_interval_summary(model_name):
    comparison_model = build(model_name, seed=SEED)
    comparison_results = run(comparison_model, districts, n_simus=N_SIMUS)
    comparison_seats = seats_by_simulation(
        comparison_results, first_round.first_round_seats
    )
    return (
        comparison_seats
        .select(["NFP+", "ENS+", "RN+"])
        .unpivot(variable_name="parti", value_name="sièges")
        .group_by("parti")
        .agg(
            pl.col("sièges").quantile(0.05).alias("p05"),
            pl.col("sièges").median().alias("médiane"),
            pl.col("sièges").quantile(0.95).alias("p95"),
        )
        .with_columns(pl.lit(model_name).alias("modèle"))
        .select("modèle", "parti", "p05", "médiane", "p95")
    )

In [16]:
comparison = pl.concat(
    [
        seat_interval_summary("national_anchored"),
        seat_interval_summary("kernel_anchored"),
    ]
).sort(["parti", "modèle"])
comparison

modèle,parti,p05,médiane,p95
str,str,f64,f64,f64
"""kernel_anchored""","""ENS+""",143.0,161.0,179.0
"""national_anchored""","""ENS+""",130.0,167.0,182.0
"""kernel_anchored""","""NFP+""",179.0,201.0,227.0
"""national_anchored""","""NFP+""",154.0,201.5,237.0
"""kernel_anchored""","""RN+""",102.0,132.0,163.0
"""national_anchored""","""RN+""",83.0,128.0,204.0


Avec seulement `N_SIMUS` tirages, cette dernière table sert à comprendre la comparaison, pas à publier des bornes précises. Les résultats finaux doivent être produits par le script de reproduction avec davantage de simulations.

Le fil logique reste toutefois le même pour les trois variantes : **premier tour → hypothèses de report → matrices plausibles → voix du second tour → sièges**.